[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Deep-unlearning/smol-audio/blob/master/cohere_transcribe_streaming/Stream_Cohere_Transcribe.ipynb)

# Stream Cohere Transcribe, live

[Cohere Transcribe](https://huggingface.co/CohereLabs/cohere-transcribe-03-2026) is a 2B speech-to-text model that
needs the **whole recording** before it can answer. **Cohere Streaming**
([`Steveeeeeeen/cohere-streaming`](https://huggingface.co/Steveeeeeeen/cohere-streaming)) is a version of it that
listens **as the audio arrives** and writes captions while you are still speaking.

In this notebook you will:

1. install the dependencies and load the streaming model,
2. download a short speech clip,
3. feed it to the model in tiny 80 ms pieces and watch the transcript grow,
4. get the final, polished transcript,
5. compare different latency settings (`lookahead` = 0, 1, 3, 6, 13),
6. (optional) measure word error rate with `jiwer`.

**Hardware.** Use a GPU runtime (Colab: *Runtime > Change runtime type > T4 GPU* or better). The model has 2B
parameters, so on a CPU it works but is much slower than real time. Loading also temporarily holds the original
offline encoder and its streaming replacement together in host memory: allow generous system RAM as well as GPU
memory. The CPU path is useful for learning the API when sufficient RAM is available.

This example is part of [smol-audio](https://github.com/Deep-unlearning/smol-audio/tree/master/cohere_transcribe_streaming), adapted from [the upstream notebook contribution](https://github.com/Deep-unlearning/cohere-transcribe-streaming/tree/1e1b41ca259b85d0e2b257b6a794d1b9c1e1ef2d) ([PR #3](https://github.com/Deep-unlearning/cohere-transcribe-streaming/pull/3)).

## 1. Setup

The code lives in the `cohere_transcribe_streaming` folder of [smol-audio](https://github.com/Deep-unlearning/smol-audio).
The cell below clones the repo in a fresh Colab runtime, or reuses your local checkout, then moves into that folder.
We install the dependencies from `requirements.txt`, plus `jiwer` (to compute word error rate later) and `pyarrow`
(to read the sample clip).

In [ ]:
from pathlib import Path
import subprocess

# Reuse a local checkout, or clone smol-audio in a fresh Colab runtime. Safe to rerun this cell.
here = Path.cwd()
if here.name == "cohere_transcribe_streaming" and (here / "model.py").is_file():
    example_dir = here
elif (here / "cohere_transcribe_streaming" / "model.py").is_file():
    example_dir = here / "cohere_transcribe_streaming"
else:
    checkout = here / "smol-audio"
    if not checkout.exists():
        subprocess.run(["git", "clone", "--quiet", "https://github.com/Deep-unlearning/smol-audio", str(checkout)], check=True)
    example_dir = checkout / "cohere_transcribe_streaming"
%cd {example_dir}
%pip install -r requirements.txt jiwer pyarrow


Let's check which device we have. We choose native bf16 on recent GPUs, fp16 on a T4, and float32 on a CPU.
A T4 can store bf16 tensors but does not have native bf16 compute support, so it uses fp16 here.

In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NATIVE_BF16 = DEVICE == "cuda" and torch.cuda.is_bf16_supported(including_emulation=False)
DTYPE = torch.bfloat16 if NATIVE_BF16 else torch.float16 if DEVICE == "cuda" else torch.float32
FINAL_REENCODE_MAX_S = 30.0  # the endpoint pass buffers only short utterances

print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "none (running on CPU, expect it to be slow)")
print("inference dtype:", DTYPE)


## 2. Get a speech clip

We use a clip from a tiny LibriSpeech test set (read English audiobooks) hosted on the Hub. We pick one short clip,
keep its reference text (what the speaker really said) and play it.

Models like this one expect **16 kHz mono** audio as float numbers between -1 and 1. LibriSpeech already is 16 kHz.
To use your own file instead, load it with `soundfile`, average the channels, and resample to 16 kHz with `librosa`
(see [`stream_file.py`](https://github.com/Deep-unlearning/smol-audio/blob/master/cohere_transcribe_streaming/stream_file.py) in this example).

In [ ]:
import io
import numpy as np
import soundfile as sf
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download
from IPython.display import Audio

SR = 16000  # the model works at 16 kHz

parquet = hf_hub_download(
    "hf-internal-testing/librispeech_asr_dummy",
    "clean/validation-00000-of-00001.parquet",
    repo_type="dataset",
)
rows = pq.read_table(parquet, columns=["audio", "text"]).to_pylist()

def decode(row):
    wav, sr = sf.read(io.BytesIO(row["audio"]["bytes"]), dtype="float32")
    assert sr == SR
    return wav

audio = decode(rows[1])
reference = rows[1]["text"]
print(f"{len(audio) / SR:.1f} seconds of audio")
print("REFERENCE:", reference)
Audio(audio, rate=SR)

## 3. Concepts in plain words

Before we run anything, three ideas.

**80 ms chunks.** During live streaming, audio arrives in small slices, and the model eats
one slice at a time. Here each slice is **80 ms** long, which is `1280` samples at 16 kHz (that is
12.5 slices per second). Think of it as the model's heartbeat: every 80 ms it takes in a bit of sound and updates its
internal memory (a cache). At the endpoint, short clips also get a fresh encoder pass for the final transcript.

**Lookahead.** Understanding a sound sometimes depends on what comes right after it (is that "ice cream" or "I scream"?).
`lookahead` is how many extra 80 ms slices of *future* audio the model waits for before it commits to its
interpretation of a moment. More lookahead means better accuracy but later captions. The encoder
waits for `(lookahead + 1) x 80 ms` of audio per update:

| lookahead | 0 | 1 | 3 | 6 | 13 |
|---|---|---|---|---|---|
| encoder audio wait | 80 ms | 160 ms | 320 ms | 560 ms | 1.12 s |

One model supports all of these. You choose at load time. These numbers describe the encoder audio wait;
caption delay also includes model compute, partial decoding, and word commitment. Actual timing depends on hardware.

**Live vs final transcript.** There are two kinds of output:

* **Live** text (`model.step(chunk)`): a running best guess, updated as audio comes in. The last word or two can still change.
* **Final** text (`model.finish()`): with the default AED partials and free final decoding, short utterances get
  a fresh encoder pass at `final_lookahead=13`, then beam search. This makes the final decode independent of the live
  `lookahead` while the buffered utterance fits `final_reencode_max_s` (30 s by default). For longer utterances, the
  model uses its streamed encoder states, so the final result can depend on the live lookahead.

The final pass takes additional time after the speaker stops. We measure it below on your hardware; its duration
also depends on clip length, transcript length and decoding settings.

## 4. Load the streaming model

`CohereStreaming(lookahead=6)` downloads two things from the Hub: the original Cohere Transcribe weights (the decoder and
tokenizer) and the streaming checkpoint (the cache-aware encoder). The first call takes a few minutes. A lookahead of 6
gives an encoder audio wait of 560 ms; visible caption timing also includes compute and decoding.

In [ ]:
from model import CohereStreaming

model = CohereStreaming(lookahead=6, device=DEVICE, dtype=DTYPE, final_reencode_max_s=FINAL_REENCODE_MAX_S)
print("device:", model.device, "| chunk size:", model.hop_ms, "ms")


## 5. Stream the clip, 80 ms at a time

The protocol has three calls:

* `model.reset(language="en")` starts a new utterance (languages: en, fr, de, es, it, pt, nl, pl),
* `model.step(chunk)` feeds one 80 ms chunk and returns the **current live transcript**,
* `model.finish()` signals the end of speech and returns the **final transcript**.

Below we cut the clip into 1280-sample chunks and loop. We only print when the text changed, together with the
position in the audio (not wall-clock time) where it changed. Watch how words appear and get corrected, for example a
partial word like "Quill" becoming "Quilter's".

In [ ]:
import time

def stream(model, audio, language="en", verbose=False):
    """Feed `audio` to `model` in 80 ms chunks. Returns a dict with the trace of live texts and the final text."""
    hop = SR * model.hop_ms // 1000  # 1280 samples
    model.reset(language=language)
    trace, shown, step_s = [], None, []
    for start in range(0, len(audio), hop):
        chunk = audio[start:start + hop]
        if len(chunk) < hop:                                   # pad the very last chunk with silence
            chunk = np.pad(chunk, (0, hop - len(chunk)))
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        text = model.step(chunk)                               # live transcript so far
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        step_s.append(time.perf_counter() - t0)
        at = (start + hop) / SR                                # seconds of audio consumed
        if text != shown:
            trace.append((at, text))
            if verbose:
                print(f"[{at:5.2f}s] {text}")
            shown = text
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    final = model.finish()                                     # end of speech -> final transcript
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    finish_s = time.perf_counter() - t0
    return dict(trace=trace, live=shown or "", final=final, step_s=step_s, finish_s=finish_s)

out = stream(model, audio, verbose=True)

## 6. The final transcript

Now compare what the model said live, what it said after `finish()`, and the reference.
`finish_s` is how long the final pass took on this machine. CUDA timing waits for queued GPU work to finish.
It varies with hardware, clip length and decoding settings.

In [ ]:
print("LIVE (last) :", out["live"])
print("FINAL       :", out["final"])
print("REFERENCE   :", reference)
print(f"final pass took {out['finish_s'] * 1000:.0f} ms")

## 7. Compare lookahead settings

`lookahead` is fixed when the model is created, so we load the model once per setting
(0, 1, 3, 6, 13) and stream the same clip. For each setting we record:

* **encoder audio wait**: `(lookahead + 1) x 80 ms`, before compute and decoding,
* **first text at**: the audio time at which the first words showed up,
* **last change at**: the audio time of the last change of the live text,
* **updates / rewrites**: how many times the live text changed, and how many of those were *rewrites* (an earlier part of
  the text changed instead of just growing at the end),
* **live text** vs **final text**.

Expect small lookahead to show words sooner, with a few more mistakes or rewrites, and large lookahead to be later but steadier.
We free the GPU memory between runs.

In [ ]:
import gc

def rewrites(trace):
    """Count updates where the previous text is not a prefix of the new one (ignoring trailing punctuation)."""
    n = 0
    for (_, a), (_, b) in zip(trace, trace[1:]):
        if not b.startswith(a.rstrip(" .,?!")):
            n += 1
    return n

del model                                   # free the lookahead-6 model before loading the others
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

results = {}
for R in [0, 1, 3, 6, 13]:
    m = CohereStreaming(lookahead=R, device=DEVICE, dtype=DTYPE, final_reencode_max_s=FINAL_REENCODE_MAX_S)
    o = stream(m, audio)
    first = next((t for t, txt in o["trace"] if txt.strip()), float("nan"))
    results[R] = dict(o, nominal_ms=(R + 1) * 80, first_text_s=first,
                      last_change_s=o["trace"][-1][0], updates=len(o["trace"]), rewrites=rewrites(o["trace"]))
    del m
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"{'R':>3} {'audio wait':>10} {'first text':>11} {'last change':>12} {'updates':>8} {'rewrites':>9}")
for R, r in results.items():
    print(f"{R:>3} {r['nominal_ms']:>8} ms {r['first_text_s']:>9.2f} s {r['last_change_s']:>10.2f} s {r['updates']:>8} {r['rewrites']:>9}")

In [ ]:
print("REFERENCE:", reference, "\n")
for R, r in results.items():
    print(f"R={R:<2} live : {r['live']}")
    print(f"     final: {r['final']}")

For this short clip, the final pass uses the same endpoint lookahead for every setting, so the **final** lines
should match (small numerical differences may occur across hardware). This applies only while the buffered clip
fits `final_reencode_max_s`, which is 30 s here. Larger live lookahead can make the **live** text steadier.

Another useful number is the compute cost per step. For sustained live use, compute must keep up with incoming audio
on average (real-time factor below 1). Individual decoding steps can take longer than 80 ms, so a real application
needs buffering. Check the average on your hardware:

In [ ]:
import numpy as np
for R, r in results.items():
    # a step with lookahead R only runs the encoder every (R+1) chunks, so look at the average over the clip
    avg = 1000 * np.mean(r["step_s"])
    print(f"R={R:<2} average {avg:6.1f} ms per 80 ms chunk -> real-time factor {avg / 80:.2f}")

## 8. (Optional) A tiny WER check

**WER** (word error rate) is the share of words that are wrong: `(substitutions + deletions + insertions) / words in the
reference`. Lower is better. We stream a few short clips at lookahead 0 and 13 and compare the last *live* text with
the *final* text. We lowercase and remove punctuation first so that "Hello," and "hello" count as equal.
This is a smoke test on a handful of clips, not a benchmark. The full evaluation code is in the
[upstream benchmark](https://github.com/Deep-unlearning/cohere-transcribe-streaming/tree/1e1b41ca259b85d0e2b257b6a794d1b9c1e1ef2d/benchmark).

In [ ]:
import re
import jiwer

def norm(s):
    return re.sub(r"[^a-z' ]", "", s.lower().replace("-", " ")).split()

decoded = [(decode(r), r["text"]) for r in rows]
clips = [(a, text) for a, text in decoded if len(a) / SR < FINAL_REENCODE_MAX_S][:4]
print(len(clips), "clips,", f"{sum(len(a) for a, _ in clips) / SR:.0f} s of audio in total")

wer_rows = {}
for R in [0, 13]:
    m = CohereStreaming(lookahead=R, device=DEVICE, dtype=DTYPE, final_reencode_max_s=FINAL_REENCODE_MAX_S)
    outs = [stream(m, a) for a, _ in clips]
    refs = [" ".join(norm(t)) for _, t in clips]
    wer_rows[R] = dict(
        live=100 * jiwer.wer(refs, [" ".join(norm(o["live"])) for o in outs]),
        final=100 * jiwer.wer(refs, [" ".join(norm(o["final"])) for o in outs]),
    )
    del m
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"{'R':>3} {'live WER %':>11} {'final WER %':>12}")
for R, w in wer_rows.items():
    print(f"{R:>3} {w['live']:>11.1f} {w['final']:>12.1f}")

## Where to go next

* **Your own audio:** run `python stream_file.py my.wav --language fr --lookahead 3 --realtime` from `smol-audio/cohere_transcribe_streaming`.
* **Real microphone / server:** call `model.step()` from your audio callback with 1280-sample float32 chunks.
* **Benchmarks:** the [upstream benchmark](https://github.com/Deep-unlearning/cohere-transcribe-streaming/tree/1e1b41ca259b85d0e2b257b6a794d1b9c1e1ef2d/benchmark) includes word-by-word delay and retraction measurements on LibriSpeech.
* **How was it trained?** See [Distill_Streaming_Student.ipynb](https://github.com/Deep-unlearning/smol-audio/blob/master/cohere_transcribe_streaming/Distill_Streaming_Student.ipynb).
